# ATL_EBD_2A Processing — v2.0

Grids **lidar ratio** and **depolarisation ratio** from the ATL_EBD_2A product onto a
global 1° × 100 m grid for December 2025.

## What is new in v2 compared to v1.5

| | v1.5 | v2 |
|---|---|---|
| Variables processed | 4 (lidar ratio, depol, backscatter, extinction) | 2 (lidar ratio + depol only) |
| SNR filter | none | keep pixel only if `value / error > 3` |
| Physical bounds filter | none | lidar ratio 0–200 sr, depol ratio 0–1 |
| Error variables | accumulated as output | used as quality filter only |
| Diagnostic test | basic 5-ZIP visual check | descriptive stats + rejection counts |

The v1.5 output contained physically impossible values (lidar ratios up to ~10¹¹ sr,
negative depol ratios) because no pixel-level quality gate existed beyond the ice mask.
The two new filters in v2 remove these bad retrievals before accumulation.

## Workflow
1. Run **Cells 0–4** to set up (imports, config, discovery, grid).
2. Run **Cell 5** to define the core functions.
3. Run **Cell 6** (diagnostic test, 5 ZIPs) and review the printed stats.
   If the stats look physically reasonable, proceed.
4. Run **Cells 7–9** to process the full batch and save output.

**Runs on:** Work PC (needs access to remote EBD and TC data).

## 0. Imports

In [ ]:
import earthcarekit as eck
from pathlib import Path

# Set the earthcarekit config if running on the work PC.
# On the local Mac the config file does not exist, so we skip this call
# (plotting notebooks that only read NetCDF outputs can run without it).
_cfg = Path("/usr/people/raucher/Documents/Config_ECK/default_config.toml")
if _cfg.exists():
    eck.set_config(str(_cfg))

import xarray as xr
import numpy as np
import re as _re
import sys, shlex, shutil
from datetime import datetime, timedelta, date
from scipy.interpolate import interp1d

# my_subprocess is a KNMI utility for running shell commands and capturing output.
# It lives in Gerd-Jan's OneDrive folder on the work PC.
MY_SUBPROCESS_FILE = Path("/usr/people/raucher/Documents/Coding1/Gerd-Jan/OneDrive_1_24-2-2026")
if str(MY_SUBPROCESS_FILE) not in sys.path:
    sys.path.insert(0, str(MY_SUBPROCESS_FILE))
from my_subprocess import run_shell_cmd_and_communicate, print_shell_output

print("Imports OK")

## 0b. ZIP helpers

The raw EarthCARE data is stored on the remote server as one ZIP per orbit, each
containing one HDF5 file. Because the remote filesystem is slow, we copy one ZIP at
a time to local scratch, process it, then delete it — never modifying remote files.

In [ ]:
def _to_date(v):
    """Convert str / datetime / date to a date object."""
    if isinstance(v, datetime): return v.date()
    if isinstance(v, date):     return v
    if isinstance(v, str):      return datetime.strptime(v, "%Y-%m-%d").date()
    raise TypeError(type(v))


def run_cmd_checked(cmd):
    """Run a shell command; raise RuntimeError if it returns a non-zero exit code."""
    lines_out, lines_err, rc = run_shell_cmd_and_communicate(cmd, verbose=False)
    if rc != 0:
        print_shell_output(lines_out, lines_err, prefix="[shell] ")
        raise RuntimeError(f"Command failed (exit {rc}): {cmd}")
    return lines_out, lines_err


def discover_remote_zip_files(root, start_date, end_date):
    """
    Walk the date-tree under `root` and return all ZIP files whose date falls
    within [start_date, end_date].  The tree structure is root/YYYY/MM/DD/.
    """
    root, start, end = Path(root), _to_date(start_date), _to_date(end_date)
    zips, day = [], start
    while day <= end:
        d = root / day.strftime("%Y") / day.strftime("%m") / day.strftime("%d")
        if d.exists():
            zips.extend(sorted(d.glob("*.ZIP")))
            zips.extend(sorted(d.glob("*.zip")))
        day += timedelta(days=1)
    return sorted(dict.fromkeys(zips))


def stage_zip_and_extract(src_zip, stage_root):
    """
    Copy a ZIP from the remote server to local scratch and extract it.
    Returns (local_zip_path, extract_dir_path, list_of_h5_files).
    """
    stage_root.mkdir(parents=True, exist_ok=True)
    local_zip   = stage_root / src_zip.name
    extract_dir = stage_root / src_zip.stem
    if local_zip.exists():   local_zip.unlink()
    if extract_dir.exists(): shutil.rmtree(extract_dir)
    run_cmd_checked(f"cp {shlex.quote(str(src_zip))} {shlex.quote(str(local_zip))}")
    run_cmd_checked(f"unzip -oq {shlex.quote(str(local_zip))} -d {shlex.quote(str(extract_dir))}")
    h5_files = sorted(extract_dir.rglob("*.h5"))
    if not h5_files:
        raise FileNotFoundError(f"No .h5 files found after extracting: {src_zip}")
    return local_zip, extract_dir, h5_files


def cleanup_staged_data(local_zip, extract_dir):
    """Delete local staged files after processing to free disk space."""
    if extract_dir is not None and extract_dir.exists(): shutil.rmtree(extract_dir)
    if local_zip   is not None and local_zip.exists():   local_zip.unlink()


_DT_FMT = "%Y%m%dT%H%M%SZ"

def _parse_times(p):
    """
    Parse the start and end timestamps from an EarthCARE filename.
    Filename format: ..._YYYYMMDDTHHMMSSZ_YYYYMMDDTHHMMSSZ_ORBITFRAME.ZIP
    Returns (start_datetime, end_datetime), or (None, None) if not found.
    """
    m = _re.search(r'_(\d{8}T\d{6}Z)_(\d{8}T\d{6}Z)_', Path(p).name)
    if not m:
        return None, None
    return datetime.strptime(m.group(1), _DT_FMT), datetime.strptime(m.group(2), _DT_FMT)


def build_tc_time_index(tc_zips):
    """
    Build a list of (start, end, path) tuples for all TC ZIP files.
    Used for time-overlap matching against EBD ZIPs.

    NOTE: EBD and TC products use different orbit numbering, so matching by
    orbit key does not work.  Time-overlap matching is used instead.
    """
    index = []
    for p in tc_zips:
        s, e = _parse_times(p)
        if s is not None:
            index.append((s, e, p))
    return index


def find_matching_tc_zip(ebd_zip, tc_index):
    """
    Return the TC ZIP whose time window most overlaps with the EBD ZIP.
    If no TC file overlaps at all, returns None.

    How it works:
      overlap = min(ebd_end, tc_end) - max(ebd_start, tc_start)
    A positive overlap means the two files cover some of the same time.
    We pick the TC file with the largest overlap (in case of multiple candidates).
    """
    ebd_start, ebd_end = _parse_times(ebd_zip)
    if ebd_start is None:
        return None
    best_path    = None
    best_overlap = timedelta(0)
    for tc_start, tc_end, tc_path in tc_index:
        overlap = min(ebd_end, tc_end) - max(ebd_start, tc_start)
        if overlap > best_overlap:
            best_overlap = overlap
            best_path    = tc_path
    return best_path


print("ZIP helpers defined")

## 1. Config

All tunable parameters live here.  Change dates and paths as needed;
do **not** change grid settings (they must match other processing notebooks
so that outputs can be compared and combined).

In [ ]:
# ── Remote data paths ────────────────────────────────────────────────────────
# ATL_EBD_2A contains the optical retrievals (extinction, backscatter, lidar ratio,
# depolarisation ratio) at the ATLID 355 nm wavelength.
REMOTE_EBD_ROOT = Path("/net/pc230016/nobackup_1/users/zadelhof/EarthCARE_DATA/L2/ATL_EBD_2A")
# ATL_TC__2A contains the target classification (what type of cloud/aerosol each pixel is).
# We need it to build the ice mask — we only want pixels classified as ice cloud (code 3).
REMOTE_TC_ROOT  = Path("/net/pc230016/nobackup_1/users/zadelhof/EarthCARE_DATA/L2/ATL_TC__2A")

# ── Local scratch directories (work PC only) ──────────────────────────────────
# ZIPs are copied here one at a time, processed, then deleted.
LOCAL_STAGE_EBD = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/temp_ZIPs_EBD")
LOCAL_STAGE_TC  = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/temp_ZIPs_TC_for_EBD")
LOCAL_STAGE_EBD.mkdir(parents=True, exist_ok=True)
LOCAL_STAGE_TC.mkdir(parents=True, exist_ok=True)

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/ATC_output/20251201_20251231")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Date range ────────────────────────────────────────────────────────────────
START_DATE = "2025-12-01"
END_DATE   = "2025-12-31"

# ── Variables to process ─────────────────────────────────────────────────────
# Dict mapping HDF5 variable name  →  short name used in output NetCDF.
# Each main variable has a paired retrieval-error variable in the HDF5 file,
# named by appending '_error'.  The error is used as a quality filter (SNR check)
# but is NOT saved to the output — only the filtered mean and std are saved.
DATA_VARS = {
    "lidar_ratio_355nm":                 "lidar_ratio",
    "particle_linear_depol_ratio_355nm": "depol_ratio",
}

# ── SNR filter ────────────────────────────────────────────────────────────────
# Signal-to-noise ratio threshold.  A pixel is accepted only if:
#   value / retrieval_error  >  SNR_MIN
# This means we keep only retrievals where the signal is at least 3× the
# uncertainty estimate.  Pixels with very noisy or uncertain retrievals are
# discarded.  Recommended by supervisor to reduce noise-driven outliers.
# Note: pixels with error ≤ 0 are also rejected (guard against division by zero
# and negative/zero error values which indicate a failed retrieval).
SNR_MIN = 3.0

# ── Physical bounds filter ────────────────────────────────────────────────────
# Hard limits on physically possible values for ice clouds.
# Values outside these ranges cannot be real ice-cloud retrievals and indicate
# retrieval failure or data corruption.  Applied AFTER the SNR filter.
#
# Lidar ratio for ice clouds: literature reports ~10–50 sr, absolute max ~200 sr.
# Linear depolarisation ratio: dimensionless, must be in [0, 1] by definition.
PHYSICAL_BOUNDS = {
    "lidar_ratio_355nm":                 (0.0, 200.0),   # units: sr
    "particle_linear_depol_ratio_355nm": (0.0,   1.0),   # dimensionless
}

# ── ATL_TC ice mask settings ──────────────────────────────────────────────────
# Classification code 3 = ice cloud in ATL_TC__2A.
TC_CLASS_VAR     = "classification"
TC_ICE_CODE      = 3
# Codes that represent ground clutter, noise, or missing data — excluded from
# the ice mask so they cannot accidentally pass as non-ice (they are not valid
# observations and should not count as 'not ice').
TC_EXCLUDE_CODES = [-2, -1, -3]
# quality_status: 0 = Good, 1 = Likely Good (degraded / lower SNR), 2+ = bad.
# We accept 0 and 1 to maximise coverage while still rejecting bad retrievals.
TC_QC_VAR        = "quality_status"
TC_MAX_QC_FLAG   = 1
HEIGHT_VAR       = "height"

# ── Grid settings ─────────────────────────────────────────────────────────────
# Must match all other v1.5/v2 processing notebooks so outputs are comparable.
GRID_RES_DEG = 1.0          # degrees, for both lat and lon
MAX_HEIGHT_M = 20_000.0     # metres — top of atmosphere for this study
HEIGHT_STEP  = 100.0        # metres — vertical resolution
MIN_SAMPLES  = 10           # minimum pixel count per cell to report a mean

# Derived grid arrays
lat_bins  = np.arange(-90.0,  90.0  + GRID_RES_DEG, GRID_RES_DEG)   # 181 edges → 180 bins
lon_bins  = np.arange(-180.0, 180.0 + GRID_RES_DEG, GRID_RES_DEG)   # 361 edges → 360 bins
target_h  = np.arange(0.0, MAX_HEIGHT_M + 1.0, HEIGHT_STEP)          # 201 levels

n_lat = len(lat_bins) - 1   # 180
n_lon = len(lon_bins) - 1   # 360
n_h   = len(target_h)       # 201

start_dt = datetime.strptime(START_DATE, "%Y-%m-%d").date()
end_dt   = datetime.strptime(END_DATE,   "%Y-%m-%d").date()

print("Config ready")
print(f"Date range : {start_dt} → {end_dt}")
print(f"Grid       : {n_lat} lat × {n_lon} lon × {n_h} height levels")
print(f"Variables  : {list(DATA_VARS.values())}")
print(f"SNR_MIN    : {SNR_MIN}")
print(f"Bounds     : { {v: PHYSICAL_BOUNDS[k] for k, v in DATA_VARS.items()} }")

## 2. File discovery

Scan the remote date-tree for all EBD and TC ZIP files in the date range,
then build a lookup dict so we can find the TC file matching each EBD file by orbit key.

In [ ]:
ebd_zips = discover_remote_zip_files(REMOTE_EBD_ROOT, START_DATE, END_DATE)
tc_zips  = discover_remote_zip_files(REMOTE_TC_ROOT,  START_DATE, END_DATE)

# Build time index for TC files (EBD and TC use different orbit numbering,
# so we match by time overlap rather than orbit key).
tc_index = build_tc_time_index(tc_zips)

# Pre-compute the EBD → TC mapping for all EBD ZIPs at once.
# This is done up front so the batch loop doesn't repeat the search each iteration.
tc_zip_for_ebd = {z: find_matching_tc_zip(z, tc_index) for z in ebd_zips}

n_matched  = sum(1 for v in tc_zip_for_ebd.values() if v is not None)
n_unmatched = len(ebd_zips) - n_matched

print(f"EBD ZIPs found  : {len(ebd_zips)}")
print(f"TC  ZIPs found  : {len(tc_zips)}")
print(f"EBD ZIPs matched to a TC file : {n_matched}")
if n_unmatched:
    print(f"WARNING: {n_unmatched} EBD ZIPs have no matching TC file and will be skipped.")

# Sanity check: print the first match so you can verify the pairing looks right
first = ebd_zips[0]
print(f"\nExample match:")
print(f"  EBD: {first.name}")
print(f"  TC : {tc_zip_for_ebd[first].name if tc_zip_for_ebd[first] else 'None'}")

## 3. Single-file inspection

Open the first EBD and TC file to verify the data structure and check that
the retrieval-error variables exist and are non-trivial.  **Run this before
the full batch** to catch any product format changes early.

In [ ]:
_first_zip = ebd_zips[0]
_first_tc_zip = tc_zip_for_ebd[_first_zip]

_local_zip, _extract_dir, _h5s = stage_zip_and_extract(_first_zip, LOCAL_STAGE_EBD)
_fp = _h5s[0]

print(f"Inspecting: {_fp.name}")
print(f"Matched TC: {_first_tc_zip.name if _first_tc_zip else 'None'}")

with eck.read_product(str(_fp)) as ds:
    print("\n── Dataset overview ──────────────────────────────")
    print(ds)

    print("\n── Optical variables + error pairs ──────────────")
    for var, short in DATA_VARS.items():
        err_var = var + "_error"
        for v in [var, err_var]:
            if v in ds.data_vars:
                arr = ds[v].values.astype(float)
                finite = arr[np.isfinite(arr)]
                nan_pct = 100 * (1 - len(finite) / arr.size)
                print(f"  {v:50s}  shape={arr.shape}  "
                      f"min={np.nanmin(arr):.3g}  max={np.nanmax(arr):.3g}  "
                      f"NaN={nan_pct:.1f}%")
            else:
                print(f"  {v:50s}  *** NOT FOUND — check product version ***")

cleanup_staged_data(_local_zip, _extract_dir)
print("\nInspection done.")

## 4. Grid setup and accumulator initialisation

We accumulate pixel values into three 3-D arrays per variable:

| Accumulator | What is summed | Used for |
|---|---|---|
| `sum`    | pixel values              | compute cell mean: `mean = sum / count` |
| `sum_sq` | squared pixel values      | compute std: `std = sqrt(sum_sq/count - mean²)` |
| `count`  | number of valid pixels    | denominator; cells with count < 10 are masked |

Using sums (rather than storing all raw values) lets us process the entire month
of data in a single pass without running out of memory.

In [ ]:
def make_accumulators():
    """
    Return a fresh dict of zero-filled accumulator arrays.
    Shape is (n_lat, n_lon, n_h) for the 3-D output,
    plus a rejection-counter dict for diagnostic reporting.
    """
    acc = {}
    for var in DATA_VARS:
        acc[var] = {
            "sum":    np.zeros((n_lat, n_lon, n_h), dtype=np.float64),
            "sum_sq": np.zeros((n_lat, n_lon, n_h), dtype=np.float64),
            "count":  np.zeros((n_lat, n_lon, n_h), dtype=np.int64),
        }
    # Rejection counters: track how many pixels are rejected by each filter.
    # Useful for the diagnostic test and for understanding data quality.
    counters = {
        "total_pixels":   0,   # all ice-masked pixels encountered
        "rejected_snr":   0,   # rejected because value/error <= SNR_MIN
        "rejected_bounds": 0,  # rejected because value outside physical bounds
        "accepted":       0,   # pixels that passed both filters and were accumulated
    }
    return acc, counters


def merge_acc(global_acc, global_counters, local_acc, local_counters):
    """
    Add the accumulators from one file into the global accumulators.
    Called after processing each HDF5 file.
    """
    for var in DATA_VARS:
        global_acc[var]["sum"]    += local_acc[var]["sum"]
        global_acc[var]["sum_sq"] += local_acc[var]["sum_sq"]
        global_acc[var]["count"]  += local_acc[var]["count"]
    for k in global_counters:
        global_counters[k] += local_counters[k]


print(f"Accumulator shape per variable: ({n_lat}, {n_lon}, {n_h})")
print(f"Memory per variable (3 arrays): "
      f"{3 * n_lat * n_lon * n_h * 8 / 1e6:.1f} MB")
print(f"Total for {len(DATA_VARS)} variables: "
      f"{3 * len(DATA_VARS) * n_lat * n_lon * n_h * 8 / 1e6:.1f} MB")

## 5. Core processing functions

Three functions make up the processing pipeline:

1. **`_build_ice_mask`** — reads the ATL_TC classification file and returns a boolean
   array indicating which (profile, height) pixels are ice cloud.
2. **`_quality_mask`** — applies the SNR filter and physical bounds filter to decide
   which pixels are good enough to include in the gridded mean.
3. **`accumulate_one_file`** — orchestrates the full pipeline for one HDF5 file:
   reads data → builds masks → interpolates → accumulates into grid cells.

In [ ]:
def _build_ice_mask(tc_fp, n_obs):
    """
    Build a boolean ice mask from a single ATL_TC__2A HDF5 file.

    Returns
    -------
    ice_mask : bool array, shape (n_obs, n_h)
        True  → pixel is classified as ice cloud AND passes quality checks.
        False → not ice, bad quality, or no valid retrieval.
    """
    ice_mask = np.zeros((n_obs, n_h), dtype=bool)

    with eck.read_product(str(tc_fp)) as ds_tc:
        tc_cls = ds_tc[TC_CLASS_VAR].values.astype(float)
        tc_h   = ds_tc[HEIGHT_VAR].values.astype(float)
        has_qc = TC_QC_VAR in ds_tc.data_vars
        tc_qc  = ds_tc[TC_QC_VAR].values.astype(int) if has_qc else None

    for i in range(min(n_obs, tc_cls.shape[0])):
        h_i   = tc_h[i, :]
        cls_i = tc_cls[i, :]
        qc_ok = (tc_qc[i, :] <= TC_MAX_QC_FLAG) if has_qc else True

        valid = (
            np.isfinite(h_i) & np.isfinite(cls_i)
            & ~np.isin(cls_i, TC_EXCLUDE_CODES)
            & qc_ok
        )
        if valid.sum() < 2:
            continue

        h_v, cls_v = h_i[valid], cls_i[valid]
        order = np.argsort(h_v)
        h_v, cls_v = h_v[order], cls_v[order]
        h_v, idx = np.unique(h_v, return_index=True)
        cls_v = cls_v[idx]
        if h_v.size < 2:
            continue

        # Nearest-neighbour interpolation: we want exact ice/not-ice per level,
        # not a blend between class codes.
        ice_v = (cls_v == TC_ICE_CODE).astype(float)
        f = interp1d(h_v, ice_v, kind="nearest", bounds_error=False, fill_value=0.0)
        ice_mask[i, :] = f(target_h).astype(bool)

    return ice_mask


def _quality_mask(value_1d, error_1d, var_name):
    """
    Return a boolean mask (shape: n_h).
    True = pixel passed both SNR filter and physical bounds filter.

    SNR filter: value / error > SNR_MIN  (error must be > 0)
    Bounds filter: lo < value < hi  (from PHYSICAL_BOUNDS)
    """
    lo, hi = PHYSICAL_BOUNDS[var_name]
    safe_error = np.where(error_1d > 0, error_1d, np.inf)
    snr_ok    = (error_1d > 0) & (value_1d / safe_error > SNR_MIN)
    bounds_ok = (value_1d > lo) & (value_1d < hi)
    return snr_ok & bounds_ok


def accumulate_one_file(ebd_fp, tc_fp, acc, counters):
    """
    Process one ATL_EBD_2A HDF5 file and add its pixels into the accumulators.
    """
    # ── Step 1: Load EBD data ────────────────────────────────────────────────
    with eck.read_product(str(ebd_fp)) as ds:
        h_raw     = ds[HEIGHT_VAR].values.astype(float)
        lat       = ds["latitude"].values
        lon       = ds["longitude"].values
        var_data  = {v: ds[v].values.astype(float) for v in DATA_VARS}
        var_error = {v: ds[v + "_error"].values.astype(float) for v in DATA_VARS}
        n_obs = h_raw.shape[0]

    # ── Step 2: Build ice mask from TC file ──────────────────────────────────
    ice_mask = _build_ice_mask(tc_fp, n_obs)   # (n_obs, n_h)

    # ── Step 3: Map profiles to grid-cell indices ────────────────────────────
    lat_idx = np.clip(np.searchsorted(lat_bins[1:-1], lat), 0, n_lat - 1)
    lon_idx = np.clip(np.searchsorted(lon_bins[1:-1], lon), 0, n_lon - 1)

    # ── Step 4: Per-variable accumulation ───────────────────────────────────
    for var in DATA_VARS:
        d_raw = var_data[var]
        e_raw = var_error[var]
        lo, hi = PHYSICAL_BOUNDS[var]

        flat_idx_list, val_list, val_sq_list = [], [], []

        for i in range(n_obs):
            if not ice_mask[i, :].any():
                continue

            h_i = h_raw[i, :]; d_i = d_raw[i, :]; e_i = e_raw[i, :]

            valid_raw = np.isfinite(h_i) & np.isfinite(d_i) & np.isfinite(e_i)
            if valid_raw.sum() < 2:
                continue

            h_v = h_i[valid_raw]; d_v = d_i[valid_raw]; e_v = e_i[valid_raw]
            order = np.argsort(h_v)
            h_v, d_v, e_v = h_v[order], d_v[order], e_v[order]
            h_v, idx_u = np.unique(h_v, return_index=True)
            d_v, e_v = d_v[idx_u], e_v[idx_u]
            if h_v.size < 2:
                continue

            # Interpolate value and error onto target height grid
            d_interp = interp1d(h_v, d_v, kind="linear",
                                bounds_error=False, fill_value=np.nan)(target_h)
            e_interp = interp1d(h_v, e_v, kind="linear",
                                bounds_error=False, fill_value=np.nan)(target_h)

            # ── Step 5: Apply ice mask ───────────────────────────────────────
            d_interp[~ice_mask[i, :]] = np.nan
            e_interp[~ice_mask[i, :]] = np.nan

            # ── Step 6: Count rejections, then apply quality mask ────────────
            # Work with boolean arrays of shape (n_h,) — no class tricks.
            is_ice     = ice_mask[i, :]                          # (n_h,) bool
            is_finite  = np.isfinite(d_interp)                   # (n_h,) bool
            safe_e     = np.where(e_interp > 0, e_interp, np.inf)
            snr_ok     = (e_interp > 0) & (d_interp / safe_e > SNR_MIN)
            bounds_ok  = (d_interp > lo) & (d_interp < hi)

            counters["total_pixels"]    += int(is_ice.sum())
            counters["rejected_snr"]    += int((is_ice & is_finite & ~snr_ok).sum())
            counters["rejected_bounds"] += int((is_ice & is_finite & snr_ok & ~bounds_ok).sum())
            counters["accepted"]        += int((is_ice & snr_ok & bounds_ok).sum())

            # Zero out pixels that failed quality checks
            d_interp[~(snr_ok & bounds_ok)] = np.nan

            # ── Step 7: Collect valid (flat index, value) pairs ──────────────
            # flat_index = lat_idx * (n_lon * n_h) + lon_idx * n_h + h_idx
            # Uniquely identifies the 3-D grid cell for each (profile, height).
            valid_h = np.where(np.isfinite(d_interp))[0]
            if len(valid_h) == 0:
                continue

            fi   = lat_idx[i] * n_lon * n_h + lon_idx[i] * n_h + valid_h
            vals = d_interp[valid_h]
            flat_idx_list.append(fi)
            val_list.append(vals)
            val_sq_list.append(vals * vals)

        # ── Step 8: Vectorised bincount accumulation ─────────────────────────
        if flat_idx_list:
            fi_all  = np.concatenate(flat_idx_list).astype(int)
            v_all   = np.concatenate(val_list)
            vsq_all = np.concatenate(val_sq_list)
            flat_size = n_lat * n_lon * n_h

            acc[var]["sum"]    += np.bincount(fi_all, weights=v_all,   minlength=flat_size).reshape(n_lat, n_lon, n_h)
            acc[var]["sum_sq"] += np.bincount(fi_all, weights=vsq_all, minlength=flat_size).reshape(n_lat, n_lon, n_h)
            acc[var]["count"]  += np.bincount(fi_all,                  minlength=flat_size).reshape(n_lat, n_lon, n_h).astype(int)


print("Core functions defined")

## 6. Diagnostic test — 5 ZIPs

Process a small sample before committing to the full batch.
Review the descriptive statistics and rejection rates printed below.

**Expected ranges for December 2025 ice clouds:**
- Lidar ratio: bulk of pixels should be 10–60 sr; max should be < 200 sr.
- Depol ratio: bulk should be 0.2–0.6; no values outside 0–1.

**If the stats look wrong:**
- Max lidar ratio >> 200 sr → lower `SNR_MIN` or check `PHYSICAL_BOUNDS`.
- Depol ratio < 0 or > 1 → `PHYSICAL_BOUNDS` not applied correctly.
- Rejection rate > 80% → `SNR_MIN` may be too aggressive; try 2.0.
- 0 pixels accepted → check TC file matching and ice mask.

Once satisfied, proceed to Cell 7 for the full batch.

In [ ]:
N_TEST_ZIPS = 5

test_acc, test_counters = make_accumulators()
test_failed = []

for src_zip in ebd_zips[:N_TEST_ZIPS]:
    local_zip = tc_local_zip = extract_dir = tc_extract_dir = None
    try:
        local_zip, extract_dir, h5_files = stage_zip_and_extract(src_zip, LOCAL_STAGE_EBD)
        tc_zip = tc_zip_for_ebd.get(src_zip)
        if tc_zip is None:
            print(f"  No TC match for {src_zip.name} — skipping")
            continue
        tc_local_zip, tc_extract_dir, tc_h5s = stage_zip_and_extract(tc_zip, LOCAL_STAGE_TC)
        tc_fp = tc_h5s[0] if tc_h5s else None
        for fp in h5_files:
            if tc_fp is None: continue
            file_acc, file_counters = make_accumulators()
            accumulate_one_file(fp, tc_fp, file_acc, file_counters)
            merge_acc(test_acc, test_counters, file_acc, file_counters)

    except Exception as ex:
        test_failed.append(str(ex))
    finally:
        cleanup_staged_data(local_zip, extract_dir)
        cleanup_staged_data(tc_local_zip, tc_extract_dir)

print(f"Diagnostic test done — {N_TEST_ZIPS} ZIPs, {len(test_failed)} failures\n")

c = test_counters
total = max(c["total_pixels"], 1)
print("── Pixel rejection summary ──────────────────────")
print(f"  Total ice pixels encountered : {c['total_pixels']:>10,}")
print(f"  Rejected by SNR filter       : {c['rejected_snr']:>10,}  ({100*c['rejected_snr']/total:.1f}%)")
print(f"  Rejected by bounds filter    : {c['rejected_bounds']:>10,}  ({100*c['rejected_bounds']/total:.1f}%)")
print(f"  Accepted                     : {c['accepted']:>10,}  ({100*c['accepted']/total:.1f}%)")
print()

print("── Descriptive statistics (from gridded means, cells with count ≥ 1) ──")
for var, short in DATA_VARS.items():
    cnt  = test_acc[var]["count"]
    s    = test_acc[var]["sum"]
    mask = cnt > 0
    if not mask.any():
        print(f"  {short}: no data accumulated")
        continue
    # Guard division: replace zero counts with 1 before dividing.
    # np.where evaluates both branches, so s/cnt would produce NaN/inf
    # for zero-count cells even though those cells are masked out afterwards.
    safe_cnt = np.where(mask, cnt, 1)
    means = np.where(mask, s / safe_cnt, np.nan)
    vals  = means[mask]
    p25, p50, p75 = np.nanpercentile(vals, [25, 50, 75])
    print(f"  {short}")
    print(f"    cells with data : {mask.sum():,}")
    print(f"    min             : {vals.min():.4g}")
    print(f"    max             : {vals.max():.4g}")
    print(f"    mean            : {vals.mean():.4g}")
    print(f"    median          : {p50:.4g}")
    print(f"    std             : {vals.std():.4g}")
    print(f"    IQR             : {p75 - p25:.4g}  (p25={p25:.4g}, p75={p75:.4g})")
    print(f"    p5 / p95        : {np.nanpercentile(vals, 5):.4g} / {np.nanpercentile(vals, 95):.4g}")
    print()

## 7. Full batch processing

Process all ZIPs in the date range.  Progress is printed every 10 ZIPs.
The accumulators are reset here so this cell is safe to re-run.

In [ ]:
global_acc, global_counters = make_accumulators()
failed = []
n_done = 0

for src_zip in ebd_zips:
    local_zip = tc_local_zip = extract_dir = tc_extract_dir = None
    try:
        local_zip, extract_dir, h5_files = stage_zip_and_extract(src_zip, LOCAL_STAGE_EBD)
        tc_zip = tc_zip_for_ebd.get(src_zip)
        if tc_zip is None:
            continue
        tc_local_zip, tc_extract_dir, tc_h5s = stage_zip_and_extract(tc_zip, LOCAL_STAGE_TC)
        tc_fp = tc_h5s[0] if tc_h5s else None

        for fp in h5_files:
            if tc_fp is None: continue
            file_acc, file_counters = make_accumulators()
            accumulate_one_file(fp, tc_fp, file_acc, file_counters)
            merge_acc(global_acc, global_counters, file_acc, file_counters)

        n_done += 1
    except Exception as ex:
        failed.append((str(src_zip), str(ex)))
    finally:
        cleanup_staged_data(local_zip, extract_dir)
        cleanup_staged_data(tc_local_zip, tc_extract_dir)

    if n_done % 10 == 0:
        c = global_counters
        print(f"  {n_done}/{len(ebd_zips)} ZIPs  |  "
              f"accepted: {c['accepted']:,}  "
              f"rejected SNR: {c['rejected_snr']:,}  "
              f"failures: {len(failed)}")

print(f"\nBatch complete: {n_done} ZIPs processed, {len(failed)} failures")
if failed:
    print("First failure:", failed[0][1])

## 8. Compute statistics

Derive mean and standard deviation from the accumulated sums.

**Formula:**
- mean = sum / count
- variance = sum_sq / count − mean²  (computational formula for variance)
- std = sqrt(max(variance, 0))  — clamp to zero to avoid tiny negative values from floating-point rounding

Cells with fewer than `MIN_SAMPLES` (=10) valid pixels are masked to NaN
to prevent noisy, under-sampled cells from appearing in the output.

In [ ]:
stats = {}

for var, short in DATA_VARS.items():
    cnt = global_acc[var]["count"].astype(float)
    s   = global_acc[var]["sum"]
    s2  = global_acc[var]["sum_sq"]

    has_data = cnt > 0
    safe_cnt = np.where(has_data, cnt, 1)   # avoid division by zero in np.where branches

    mean     = np.where(has_data, s  / safe_cnt, np.nan)
    variance = np.where(has_data, s2 / safe_cnt - mean ** 2, np.nan)
    std      = np.sqrt(np.maximum(variance, 0.0))
    std      = np.where(cnt > 1, std, np.nan)   # std undefined for single sample

    # Mask under-sampled cells
    mean = np.where(cnt >= MIN_SAMPLES, mean, np.nan)
    std  = np.where(cnt >= MIN_SAMPLES, std,  np.nan)

    stats[var] = {"mean": mean, "std": std, "count": global_acc[var]["count"]}

    valid_cells = int(np.sum(cnt >= MIN_SAMPLES))
    print(f"{short}: {valid_cells:,} cells with ≥{MIN_SAMPLES} samples  |  "
          f"mean range [{np.nanmin(mean):.3g}, {np.nanmax(mean):.3g}]")

print("\nStatistics computed.")

## 9. Save to NetCDF

Two output files per run:
- **`_3d.nc`** — full lat × lon × height grid.  Use for map plots.
- **`_latheight.nc`** — zonal mean (lat × height).  Use for latitude-height cross-sections.

The zonal mean is computed by summing the raw accumulators along the lon axis
**before** dividing — this gives a pixel-count-weighted zonal mean, which is
mathematically correct.  Averaging the 3-D mean field along longitude would
give the wrong answer if cells have different sample counts.

Filter settings (SNR_MIN, PHYSICAL_BOUNDS) are saved as global attributes
so the output file is self-documenting.

In [ ]:
lat_centres = (lat_bins[:-1] + lat_bins[1:]) / 2
lon_centres = (lon_bins[:-1] + lon_bins[1:]) / 2

global_attrs = {
    "description":             "ATL_EBD_2A ice cloud optical properties, v2 processing",
    "date_range":              f"{START_DATE} to {END_DATE}",
    "grid_resolution_deg":     GRID_RES_DEG,
    "min_samples":             MIN_SAMPLES,
    "snr_min":                 SNR_MIN,
    "bounds_lidar_ratio_sr":   str(PHYSICAL_BOUNDS["lidar_ratio_355nm"]),
    "bounds_depol_ratio":      str(PHYSICAL_BOUNDS["particle_linear_depol_ratio_355nm"]),
    "pixels_accepted":         int(global_counters["accepted"]),
    "pixels_rejected_snr":     int(global_counters["rejected_snr"]),
    "pixels_rejected_bounds":  int(global_counters["rejected_bounds"]),
}

tag = f"{START_DATE.replace('-', '')}_{END_DATE.replace('-', '')}"

UNITS = {
    "lidar_ratio_355nm":                 "sr",
    "particle_linear_depol_ratio_355nm": "1",
}

def make_da_3d(data, long_name, units):
    return xr.DataArray(data.astype(np.float32),
                        dims=["latitude", "longitude", "height"],
                        attrs={"long_name": long_name, "units": units})

def make_da_lh(data, long_name, units):
    return xr.DataArray(data.astype(np.float32),
                        dims=["latitude", "height"],
                        attrs={"long_name": long_name, "units": units})

data_vars_3d = {}
data_vars_lh = {}

for var, short in DATA_VARS.items():
    u = UNITS[var]

    data_vars_3d[f"{short}_mean"]  = make_da_3d(stats[var]["mean"],                  f"{short} cell mean",            u)
    data_vars_3d[f"{short}_std"]   = make_da_3d(stats[var]["std"],                   f"{short} standard deviation",   u)
    data_vars_3d[f"{short}_count"] = make_da_3d(stats[var]["count"].astype(float),   f"{short} pixel count per cell", "1")

    # Zonal mean: sum accumulators along lon axis BEFORE dividing.
    # This gives a pixel-count-weighted zonal mean (mathematically correct).
    # Do NOT average the 3-D mean field along lon — that would be wrong
    # when cells have different sample counts.
    lh_sum    = global_acc[var]["sum"].sum(axis=1)            # (n_lat, n_h)
    lh_sum_sq = global_acc[var]["sum_sq"].sum(axis=1)
    lh_count  = global_acc[var]["count"].sum(axis=1).astype(float)

    lh_has      = lh_count > 0
    lh_safe_cnt = np.where(lh_has, lh_count, 1)              # guard division by zero

    lh_mean = np.where(lh_has, lh_sum    / lh_safe_cnt, np.nan)
    lh_var  = np.where(lh_has, lh_sum_sq / lh_safe_cnt - lh_mean ** 2, np.nan)
    lh_std  = np.sqrt(np.maximum(lh_var, 0.0))

    lh_mean = np.where(lh_count >= MIN_SAMPLES, lh_mean, np.nan)
    lh_std  = np.where(lh_count >= MIN_SAMPLES, lh_std,  np.nan)

    data_vars_lh[f"{short}_mean"]  = make_da_lh(lh_mean,  f"{short} zonal mean",                     u)
    data_vars_lh[f"{short}_std"]   = make_da_lh(lh_std,   f"{short} zonal standard deviation",       u)
    data_vars_lh[f"{short}_count"] = make_da_lh(lh_count, f"{short} zonal pixel count per lat band", "1")

coords_3d = {
    "latitude":  xr.DataArray(lat_centres, dims=["latitude"],  attrs={"units": "degrees_north"}),
    "longitude": xr.DataArray(lon_centres, dims=["longitude"], attrs={"units": "degrees_east"}),
    "height":    xr.DataArray(target_h,    dims=["height"],    attrs={"units": "m"}),
}
coords_lh = {"latitude": coords_3d["latitude"], "height": coords_3d["height"]}

out_3d = OUTPUT_DIR / f"ATL_EBD_2A_v2_{GRID_RES_DEG}deg_{tag}_3d.nc"
out_lh = OUTPUT_DIR / f"ATL_EBD_2A_v2_{GRID_RES_DEG}deg_{tag}_latheight.nc"

xr.Dataset(data_vars_3d, coords=coords_3d, attrs=global_attrs).to_netcdf(out_3d)
xr.Dataset(data_vars_lh, coords=coords_lh, attrs=global_attrs).to_netcdf(out_lh)

print(f"Saved 3D output       : {out_3d}")
print(f"Saved latheight output : {out_lh}")